# 5.3) (Exercise) Physically-Informed Climate Modeling

:::{admonition} Learning objectives
:class: tip

By the end of this exercise, you will:

1. Understand how using physical knowledge to rescale a machine learning model's inputs can make it more robust and generalizable,
2. Know how to use *custom data generators* to nonlinearly rescale inputs before feeding them to a neural network, and
3. Practice parameterization on a realistic research case.
:::

While this notebook's completion time may widely vary depending on your programming experience, we estimate it will take a minimum of 30 minutes and much longer if you decide to explore the source code. This notebook provides a minimal reproducible example of the work described in the following preprint:

[Beucler, Tom, Michael Pritchard, Janni Yuval, Ankitesh Gupta, Liran Peng, Stephan Rasp, Fiaz Ahmed et al. "Climate-invariant machine learning."](https://arxiv.org/abs/2112.08440),

and uses a reduced version of the data: two climate simulations of about 1.7 GiB each, downloaded once the first time you run the data cell. Questions Q1 and Q2 hold these arrays in memory, which takes about 6 GB of RAM at the peak and fits a free Colab session.

We will be relying on PyTorch, whose documentation you can find [here](https://docs.pytorch.org/docs/stable/index.html). The notebook was designed to be run on Google Colab (Google Colab tutorial [at this link](https://colab.research.google.com/drive/16pBJQePbqkz3QFV54L4NIkOn1kwpuRrj)).

While everything can be run locally and there are only a handful of lines that use Google specific libraries, we encourage beginners to use Google Colab not to run into [Python virtual environment](https://docs.python.org/3/tutorial/venv.html) issues.

Before we get started, if you are struggling with some of the exercises, do not hesitate to:

*   Use a direct Internet search, or [stackoverflow](https://stackoverflow.com/)
*   Debug your program, e.g. by following [this tutorial](https://swcarpentry.github.io/python-novice-inflammation/11-debugging/index.html)
*   Use assertions, e.g. by following [this tutorial](https://swcarpentry.github.io/python-novice-inflammation/10-defensive/index.html)
*   Ask for help on the course Forum

![Cumulonimbus cloud over a plain](_static/5.3-storm-john-fowler-unsplash.jpg)

Storms rapidly transport heat and water in the atmosphere, regulating the Earth's climate. Can you predict how storms affect atmospheric temperatures using deep learning, even in a changing climate?

*Source: Photo by <a href="https://unsplash.com/@wildhoney?utm_source=unsplash&utm_medium=referral&utm_content=creditCopyText">John Fowler</a> licensed under the <a href='https://unsplash.com/license'>Unsplash License.</a>*

## Part I: Configuration and Requirements

In [ ]:
import cartopy
import cartopy.feature as cfeature
import cartopy.crs as ccrs
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
from scipy.integrate import cumulative_trapezoid

import netCDF4
import pooch
import torch
import torch.nn as nn
import xarray as xr

# To make this notebook's output stable across runs
rnd_seed = 42
torch.manual_seed(rnd_seed)

In [ ]:
fz = 15 # Here we define the fontsize
lw = 2 # the linewidth
siz = 75 # and the scattered dots' size

plt.rc('text', usetex=False)
mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['font.family'] = 'STIXGeneral'
plt.rc('font', family='serif', size=fz)
mpl.rcParams['lines.linewidth'] = lw

In [ ]:
# Download the two climate simulations (about 1.7 GiB each) and the normalization constants.
# pooch caches every file and checks its sha256, so the download runs once.
# The two simulations sit on a SharePoint link that its owner can revoke; the hash makes a dead
# link fail loudly instead of saving a sign-in page as a data file.
path_data = "https://unils-my.sharepoint.com/:u:/g/personal/tom_beucler_unil_ch/"
cold_open = pooch.retrieve(
    path_data + "EfHoI_pZY3xAi4bLEuDobaUBjyQmoJd1AvYnoPdH01VN-w?download=1",
    known_hash="sha256:7b793afdd866a2e9b0db8fdb5029a88d557bf98525601275f5a335e95b26ac1a",
    fname="climate_invariant_cold_M4K.nc", path=pooch.os_cache("mlees"))
warm_open = pooch.retrieve(
    path_data + "Eeq_n6Qv0jZBuRkICaOb0VQB6J1cN7muM6MrA3zA-v7LFg?download=1",
    known_hash="sha256:211db8ae89904f1fa3e2f17dc623bc6f5c6156cf24f4e3a42d92660ab1790fd4",
    fname="climate_invariant_warm_P4K.nc", path=pooch.os_cache("mlees"))
cold_data = xr.open_dataset(cold_open)
warm_data = xr.open_dataset(warm_open)

# Normalization constants (mean, standard deviation, minimum, maximum of every variable),
# computed over the cold climate for the raw inputs and for each rescaled input,
# and the coefficients of the model's hybrid vertical grid
repo_data = "https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-III/"
def fetch(name, known_hash):
    return pooch.retrieve(repo_data + name, known_hash="sha256:" + known_hash,
                          fname=name, path=pooch.os_cache("mlees"))

norm_raw_open = fetch("climate_invariant_norm_raw.nc", "ee3c669928031af1a03ec3bc61373107575173decf66ede9b0c3b8568214ca0f")
RH_open = fetch("climate_invariant_norm_RH.nc", "4d5275746eb1aad4a2279e16784befaa4beeab5a2aa6545e0e85437c8d73476f")
B_open = fetch("climate_invariant_norm_BMSE.nc", "396df61a24f6111acc1b908cdda3d10e0649d3eb551de860b3ebeb4419adc514")
LHFnsDELQ_open = fetch("climate_invariant_norm_LHF_nsDELQ.nc", "514413a6ab0f33039df5f815a925cf8916454d288f384615050131f1bdc8b06f")
grid = np.load(fetch("climate_invariant_hyam_hybm.npz", "760ea10734da9382d0c367407ef0469ba651dd96f3d4b6290a8736b093038fca"))
hyam, hybm = grid["hyam"], grid["hybm"]

# Extract the range of possible longitude and latitude in case of need
longitude = cold_data.lon[:144].values
latitude = cold_data.lat[:96*144:144].values
# SPCAM's background pressure coordinates (in hPa)
pressure_levels = np.array([ 3.643466, 7.59482 , 14.356632, 24.61222 ,
                            38.2683  , 54.59548 , 72.012451,  87.82123 ,
                            103.317127, 121.547241, 142.994039, 168.22508 ,
                            197.908087, 232.828619, 273.910817, 322.241902,
                            379.100904, 445.992574, 524.687175, 609.778695,
                            691.38943 , 763.404481, 820.858369, 859.534767,
                            887.020249, 912.644547, 936.198398, 957.48548 ,
                            976.325407, 992.556095])

## Part II: Visualizing the Extrapolation Problem

We've now extracted our cold simulation data as an [Xarray Dataset](https://docs.xarray.dev/en/stable/generated/xarray.Dataset.html) called `cold_data` and our 8K-warmer simulation data as another [Xarray Dataset](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.html) called `warm_data`.

Just in case you need it, we've also extracted all possible latitude values in `latitude` and all possible longitude values in `longitude`. The vertical pressure levels are given by `pressure_levels` (in hPa).

We will soon visualize the data to give you more intuition about the prediction problem. We aim to predict the effect of ~5km-scale storm, clouds, and turbulence on the climate from the climate conditions (specific humidity `QBP` in kg/kg, temperature `TBP` in K, surface pressure `PS` in Pa, solar insolation `SOLIN` in $W/m^{2}$, surface sensible heat fluxes `SHFLX` in $W/m^{2}$ and surface latent heat fluxes `LHFLX` in $W/m^{2}$).

As an example, we'll focus on predicting "subgrid heating tendencies" or `TPHYSTND` in $K/s$ at the model's vertical levels, which is the rate at which these storms, clouds, and turbulence warm up the atmosphere. However, you could reproduce the example below with `PHQ`, which contains the "subgrid *moistening* tendencies".

Therefore, our prediction problem can be mathematically phrased as a regression problem, in which we are trying to predict:  

`y = [ TPHYSTND[:30] ]`

from

`x = [ QBP[:30] , TBP[:30] , PS , SOLIN , SHFLX , LHFLX ]`,

where we use a vertical grid with 30 vertical levels, which means that the profiles of specific humidity `QBP` and temperature `TBP` both have 30 vertical levels, while the other inputs are scalars.

![Inputs and outputs of the parameterization](_static/5.3-parameterization-inputs-outputs.png)

*The inputs or features (left) of a machine-learning parameterization are variables representing the large-scale climate properties, while the outputs or targets (right) are the rate at which storm-scale turbulence redistributes heat, moisture, and affects radiative fluxes.*

In [ ]:
cold_q_m = {}; warm_q_m = {};
dictionary = ['mean','max','min']

for idic,m in enumerate(dictionary):
    cold_q_m[m] = np.zeros((len(latitude),len(longitude)))
    warm_q_m[m] = np.zeros((len(latitude),len(longitude)))

Nsample = (len(longitude)*len(latitude)) # Total number of samples

# First, convert the arrays to numpy to accelerate calculations
# and extract the specific humidity profile q
cold_q = cold_data['vars'][:,29].values
warm_q = warm_data['vars'][:,29].values

# We will resphape both arrays to take advantage of numpy's design
# to perform operations on entire data arrays in a single opertaion
# which is faster than (nested) loops

# Count the number of world maps (timesteps) in each dataset
Nt_cold, Nt_warm = len(cold_q) // Nsample, len(warm_q) // Nsample

# Deduct the target shape of both reshaped arrays
coldq_shape = (Nt_cold, len(latitude), len(longitude))
warmq_shape = (Nt_warm, len(latitude), len(longitude))

# Reshape both arrays
# (Eliminate incomplete world maps of q at the end of the cold_q,warm_q arrays)
cold_q_reshaped = cold_q[:np.prod(coldq_shape)].reshape(coldq_shape)
warm_q_reshaped = warm_q[:np.prod(warmq_shape)].reshape(warmq_shape)

# Calculate means, mins, maxes along the first axis
cold_q_m['mean'] = np.mean(cold_q_reshaped, axis=0)
warm_q_m['mean'] = np.mean(warm_q_reshaped, axis=0)

cold_q_m['min'] = np.min(cold_q_reshaped, axis=0)
warm_q_m['min'] = np.min(warm_q_reshaped, axis=0)

cold_q_m['max'] = np.max(cold_q_reshaped, axis=0)
warm_q_m['max'] = np.max(warm_q_reshaped, axis=0)

In [ ]:
def Input_map(cold_variable, warm_variable, cold_name, warm_name, var_name,
              vmin, vmax):
    '''
    Plots maps of the cold_variable (with the title "cold_name")
    next to the warm_variable (with the title "warm_name")
    and sets the colorbar's label to "var_name".
    The colorbar ranges from vmin to vmax.
    '''
    fig, ax = plt.subplots(1,2,
                        subplot_kw={'projection':ccrs.Robinson(central_longitude=180)},
                        figsize=(12.5,6))

    cold = ax[0].pcolormesh(longitude, latitude, cold_variable,
                            transform=ccrs.PlateCarree(),
                            vmin=vmin,vmax=vmax,shading='gouraud')
    ax[0].set_title(cold_name)
    warm = ax[1].pcolormesh(longitude, latitude, warm_variable,
                            transform=ccrs.PlateCarree(),
                            vmin=vmin,vmax=vmax,shading='gouraud')
    ax[1].set_title(warm_name)

    for iplot in range(2):
        ax[iplot].coastlines(linewidth=2.0,edgecolor='0.25')
        ax[iplot].add_feature(cfeature.BORDERS,linewidth=0.5,edgecolor='0.25')

    cbar_ax = fig.add_axes([0.94,0.2,0.01,0.6])
    cbar = fig.colorbar(warm, label=var_name,cax=cbar_ax)
    cbar_ax.yaxis.set_ticks_position('right')
    cbar_ax.yaxis.set_label_position('right')

    return fig,ax

In [ ]:
Input_map(1e3*cold_q_m['mean'], 1e3*warm_q_m['mean'],
          '(Cold climate) Mean Input', '(Warm climate) Mean Input',
          'Near-surface specific humidity (g/kg)', 0, 30);

In [ ]:
Input_map(1e3*cold_q_m['min'], 1e3*warm_q_m['min'],
          '(Cold climate) Minimum Input', '(Warm climate) Minimum Input',
          'Near-surface specific humidity (g/kg)', 0, 30);

In [ ]:
Input_map(1e3*cold_q_m['max'], 1e3*warm_q_m['max'],
          '(Cold climate) Maximum Input', '(Warm climate) Maximum Input',
          'Near-surface specific humidity (g/kg)', 0, 30);

As you can see, the warm climate contains values of the inputs (here, near-surface specific humidity) that were never seen during training. This means that even if we learn an excellent machine-learning parameterization in the cold climate, using this same machine-learning parameterization in a warm climate will be a challenging *extrapolation* problem.

Two goals of this notebook will be to:

1.   Expose the failure of a deep learning parameterization when generalizing from a cold climate to a warm climate.
2.   Physically rescaling the inputs to minimize distribution changes across climates. This will improve the generalization ability of our deep learning algorithms.  

☁ Let's start right away with the first physical rescaling: specific humidity into relative humidity ☁

## **Q1) To avoid the extrapolation problems visualized above, transform specific humidity into relative humidity**

You may use the equations below, which are adapted from the [System for Atmospheric Modeling model](https://agupubs.onlinelibrary.wiley.com/doi/full/10.1029/2021MS002968) for consistency with our storm-resolving data. Assuming you already calculated the saturation pressure of water vapor over liquid water $e_{liq}$ and the saturation pressure of water vapor over ice $e_{ice}$, the model calculates relative humidity in two steps.

1.   It assumes that the system's combined saturation pressure is:

*    Equal to $e_{liq}$ for above-freezing temperatures ($T>273.16K$)
*    Equal to $e_{ice}$ for cold temperatures ($T<253.16K$)
*    A linear combination of the two in the intermediate range:

$e_{sat}=\omega \times e_{liq} + (1-\omega) \times e_{ice}$

where the weight $\omega$ is defined as:

$\omega = \frac{T-253.16K}{273.16K-253.16K}$.

2.    It then combines the ideal gas law and Dalton's law with the definition of relative humidity to calculate relative humidity as:

$RH = \frac{e}{e_{sat}} = \frac{R_v}{R_d} \times \frac{p}{e_{sat}} \times q_v$

where:

*    $R_v \approx 461 J kg^{-1} K^{-1}$ is the specific ideal gas constant for water vapor
*    $R_d \approx 287 J kg^{-1} K^{-1}$ is the specific ideal gas constant for a standard dry air mixture
*    $p$ is air pressure
*    $q_v$ is specific humidity, or equivalently the water vapor mass concentration (in kg/kg)

In [ ]:
# Assume you have access to specific humidity, temperature, and air pressure
# Remember Python indices are left-inclusive and right-exclusive
# (You can just run this cell)

specific_humidity = cold_data['vars'][:,:30] # in kg/kg
temperature = cold_data['vars'][:,30:60] # in K

P0 = 1e5 # Mean surface air pressure (Pa)
near_surface_air_pressure = cold_data['vars'][:,60]
# Formula to calculate air pressure (in Pa) using the hybrid vertical grid
# coefficients at the middle of each vertical level: hyam and hybm
air_pressure_Pa = np.outer(near_surface_air_pressure**0,P0*hyam) + \
np.outer(near_surface_air_pressure,hybm)

Our goal is to calculate relative humidity using the above equations. We'll assume we already have functions (below) giving us `e_{liq}` and `e_{ice}` as a function of air temperature `T`. These functions can be called using:

*    `eliq(temperature)`
*    `eice(temperature)`

In [ ]:
def eliq(T):
    """
    Function taking temperature (in K) and outputting liquid saturation
    pressure (in hPa) using a polynomial fit
    """
    a_liq = np.array([-0.976195544e-15,-0.952447341e-13,0.640689451e-10,
                              0.206739458e-7,0.302950461e-5,0.264847430e-3,
                              0.142986287e-1,0.443987641,6.11239921]);
    c_liq = -80
    T0 = 273.16
    return 100*np.polyval(a_liq,np.maximum(c_liq,T-T0))

def eice(T):
    """
    Function taking temperature (in K) and outputting ice saturation
    pressure (in hPa) using a polynomial fit
    """
    a_ice = np.array([0.252751365e-14,0.146898966e-11,0.385852041e-9,
                      0.602588177e-7,0.615021634e-5,0.420895665e-3,
                      0.188439774e-1,0.503160820,6.11147274]);
    c_ice = np.array([273.15,185,-100,0.00763685,0.000151069,7.48215e-07])
    T0 = 273.16
    return (T>c_ice[0])*eliq(T)+\
    (T<=c_ice[0])*(T>c_ice[1])*100*np.polyval(a_ice,T-T0)+\
    (T<=c_ice[1])*100*(c_ice[3]+np.maximum(c_ice[2],T-T0)*\
                       (c_ice[4]+np.maximum(c_ice[2],T-T0)*c_ice[5]))

Hints:



*   You can often accelerate your calculations by converting [xarray DataArray](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.html) into [numpy nd-arrays](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.html) using their [`values` method](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.values.html). You will load these values into memory when you do this, so it will increase RAM usage.
*   To implement $e_{sat}$, you may e.g., use [numpy's `where` function](https://numpy.org/doc/stable/reference/generated/numpy.where.html) or booleans.
*   Given that super-saturation is rare at climate timescales, you can optionally bound your relative humidity calculation by 1.

💡 **For all questions, you can write your own code or complete the proposed code by replacing the underscores with the appropriate script**

In [ ]:
# Here's an empty code cell to look at the data, etc.
# You can add or remove code and text cells via the "Insert" menu

In [ ]:
# Q1.1) Calculate the combined saturation water vapor pressure here

omega = (temperature - ___) / (___ - ___)

# Make sure your weight omega is always between 0 and 1
omega = ___

esat =  ___

In [ ]:
# Q1.2) Calculate relative humidity here

relative_humidity = ___

:::{admonition} **A possible solution for Q1**
:class: note dropdown

```python
# 1) Calculating saturation water vapor pressure
T0 = 273.16 # Freezing temperature in standard conditions
T00 = 253.16 # Temperature below which we use e_ice
omega = (temperature - T00) / (T0 - T00)
omega = np.maximum( 0, np.minimum( 1, omega ))

esat =  omega * eliq(temperature) + (1-omega) * eice(temperature)

# 2) Calculating relative humidity
Rd = 287 # Specific gas constant for dry air
Rv = 461 # Specific gas constant for water vapor

# We use the `values` method to convert Xarray DataArray into Numpy ND-Arrays
relative_humidity = Rv/Rd * air_pressure_Pa/esat.values * specific_humidity.values
```
:::

## **Q2) Ensure that you mostly solved the extrapolation problem visualized in Part I**

For this purpose, let's repeat the visualization of near-surface humidity, but this time using **near-surface** *relative* instead of *specific* humidity.

In [ ]:
def RH_from_climate(data):

    # 0) Extract specific humidity, temperature, and air pressure
    specific_humidity = data['vars'][:,:30] # in kg/kg
    temperature = data['vars'][:,30:60] # in K

    P0 = 1e5 # Mean surface air pressure (Pa)
    near_surface_air_pressure = data['vars'][:,60]
    # Formula to calculate air pressure (in Pa) using the hybrid vertical grid
    # coefficients at the middle of each vertical level: hyam and hybm
    air_pressure_Pa = np.outer(near_surface_air_pressure**0,P0*hyam) + \
    np.outer(near_surface_air_pressure,hybm)

    # 1) Calculating saturation water vapor pressure
    T0 = 273.16 # Freezing temperature in standard conditions
    T00 = 253.16 # Temperature below which we use e_ice
    omega = (temperature - T00) / (T0 - T00)
    omega = np.maximum( 0, np.minimum( 1, omega ))

    esat =  omega * eliq(temperature) + (1-omega) * eice(temperature)

    # 2) Calculating relative humidity
    Rd = 287 # Specific gas constant for dry air
    Rv = 461 # Specific gas constant for water vapor

    # We use the `values` method to convert Xarray DataArray into Numpy ND-Arrays
    return Rv/Rd * air_pressure_Pa/esat.values * specific_humidity.values

... so that we can calculate relative humidity for both cold and warm climates. We can then extract **near-surface** (993hPa) humidity, which is the input variable that was clearly out-of-distribution when we used *specific* humidity.

In [ ]:
RH_cold = RH_from_climate(cold_data) # Relative humidity for the cold simulation output
RH_warm = RH_from_climate(warm_data) # Relative humidity for the warm simulation output

near_surf_RH_cold = RH_cold[:,-1] # Near-surface relative humidity for the cold case
near_surf_RH_warm = RH_warm[:,-1] # Near-surface relative humidity for the warm case

In [ ]:
# First, calculate the spatial statistics of this new near-surface
# relative humidity field. You may complete the code below or write your own.
cold_RH_m = {}; warm_RH_m = {};
dictionary = ['mean','max','min']

for idic,m in enumerate(dictionary):
    cold_RH_m[m] = np.zeros((len(latitude),len(longitude)))
    warm_RH_m[m] = np.zeros((len(latitude),len(longitude)))

cold_RH_reshaped = near_surf_RH_cold[:np.prod(coldq_shape)].reshape(coldq_shape)
warm_RH_reshaped = near_surf_RH_warm[:np.prod(warmq_shape)].reshape(warmq_shape)

# Complete the code below calculating the spatial statistics
cold_RH_m['mean'] = np.___(___, axis=___)
warm_RH_m['mean'] = np.___(___, axis=___)

cold_RH_m['min'] = ___
warm_RH_m['min'] = ___

cold_RH_m['max'] = ___
warm_RH_m['max'] = ___

In [ ]:
# Visualize the mean near-surface relative humidity at each location
# You may use the Input_map function provided above
Input_map(___,___,___,___,___,___,___);

In [ ]:
# Visualize the maximum at each location

In [ ]:
# Visualize the minimum at each location

:::{admonition} **A possible solution for Q2**
:class: note dropdown

```python
# Calculate relative humidity statistics in cold/warm simulation output
cold_RH_m = {}; warm_RH_m = {};
dictionary = ['mean','max','min']

for idic,m in enumerate(dictionary):
    cold_RH_m[m] = np.zeros((len(latitude),len(longitude)))
    warm_RH_m[m] = np.zeros((len(latitude),len(longitude)))

# Calculate the spatial statistics of the newly derived RH field
cold_RH_reshaped = near_surf_RH_cold[:np.prod(coldq_shape)].reshape(coldq_shape)
warm_RH_reshaped = near_surf_RH_warm[:np.prod(warmq_shape)].reshape(warmq_shape)

cold_RH_m['mean'] = np.mean(cold_RH_reshaped, axis=0)
warm_RH_m['mean'] = np.mean(warm_RH_reshaped, axis=0)

cold_RH_m['min'] = np.min(cold_RH_reshaped, axis=0)
warm_RH_m['min'] = np.min(warm_RH_reshaped, axis=0)

cold_RH_m['max'] = np.max(cold_RH_reshaped, axis=0)
warm_RH_m['max'] = np.max(warm_RH_reshaped, axis=0)

# Visualize the new relative humidity input
Input_map(1e2*cold_RH_m['mean'], 1e2*warm_RH_m['mean'],
          '(Cold climate) Mean Input', '(Warm climate) Mean Input',
          'Relative Humidity (%)', 0, 100);
Input_map(1e2*cold_RH_m['max'], 1e2*warm_RH_m['max'],
          '(Cold climate) Max Input', '(Warm climate) Max Input',
          'Relative Humidity (%)', 0, 100);
Input_map(1e2*cold_RH_m['min'], 1e2*warm_RH_m['min'],
          '(Cold climate) Min Input', '(Warm climate) Min Input',
          'Relative Humidity (%)', 0, 100);
```
:::

😃 It looks like our input distribution is now quite similar in the cold and warm climates! Since the support (range) of the relative humidity is $\approx [0,1]$, we now expect to have mostly converted a difficult *extrapolation* case into an easier *interpolation* case. The rest of this notebook will explore the consequences of this **physical rescaling** for the performance and generalization ability of neural networks 🧠

It would take much more than the allotted 30 minutes to repeat the same physical rescaling in a data generator (also called data loader or data pipeline), so we directly give you the source code below. Run it if you would like to proceed, and read it if you want to dive into fun implementation details 🤓

A data generator reads the training set one batch at a time and hands each batch to the model. Ours is a PyTorch `Dataset` in which every item is already a whole batch: it reads one batch from the file, normalizes it, applies the physical rescalings you ask for, and returns the inputs and outputs as tensors.

![A custom data generator rescales inputs before they reach the model](_static/5.3-data-generator-diagram.png)

*Our custom data generator rescales the inputs $x$ to $\widetilde{x}$ before feeding them to the machine learning model for training.*

In [ ]:
# Source code for the moist thermodynamics library: run to proceed and read it if you are curious
# Constants for the Community Atmosphere Model
L_V = 2.501e6   # Latent heat of vaporization (J/kg)
C_P = 1.00464e3 # Specific heat capacity of air at constant pressure (J/kg/K)
G = 9.80616     # Gravitational acceleration (m/s^2)
P0 = 1e5        # Reference surface air pressure (Pa)

# Moist thermodynamics library in numpy
class ThermLibNumpy:
    @staticmethod
    def eliqNumpy(T):
        a_liq = np.float32(np.array([-0.976195544e-15,-0.952447341e-13,\
                                     0.640689451e-10,\
                          0.206739458e-7,0.302950461e-5,0.264847430e-3,\
                          0.142986287e-1,0.443987641,6.11239921]));
        c_liq = np.float32(-80.0)
        T0 = np.float32(273.16)
        return np.float32(100.0)*np.polyval(a_liq,np.maximum(c_liq,T-T0))

    @staticmethod
    def eiceNumpy(T):
        a_ice = np.float32(np.array([0.252751365e-14,0.146898966e-11,0.385852041e-9,\
                          0.602588177e-7,0.615021634e-5,0.420895665e-3,\
                          0.188439774e-1,0.503160820,6.11147274]));
        c_ice = np.float32(np.array([273.15,185,-100,0.00763685,0.000151069,7.48215e-07]))
        T0 = np.float32(273.16)
        return np.where(T>c_ice[0],ThermLibNumpy.eliqNumpy(T),\
                       np.where(T<=c_ice[1],np.float32(100.0)*(c_ice[3]+np.maximum(c_ice[2],T-T0)*\
                       (c_ice[4]+np.maximum(c_ice[2],T-T0)*c_ice[5])),\
                               np.float32(100.0)*np.polyval(a_ice,T-T0)))

    @staticmethod
    def esatNumpy(T):
        T0 = np.float32(273.16)
        T00 = np.float32(253.16)
        omtmp = (T-T00)/(T0-T00)
        omega = np.maximum(np.float32(0.0),np.minimum(np.float32(1.0),omtmp))

        return np.where(T>T0,ThermLibNumpy.eliqNumpy(T),np.where(T<T00,ThermLibNumpy.eiceNumpy(T),(omega*ThermLibNumpy.eliqNumpy(T)+(1-omega)*ThermLibNumpy.eiceNumpy(T))))

    @staticmethod
    def qvNumpy(T,RH,P0,PS,hyam,hybm):
        R = np.float32(287.0)
        Rv = np.float32(461.0)
        p = P0 * hyam + PS[:, None] * hybm # Total pressure (Pa)

        T = T.astype(np.float32)
        if type(RH) == int:
            RH = T**0
        RH = RH.astype(np.float32)
        p = p.astype(np.float32)

        return R*ThermLibNumpy.esatNumpy(T)*RH/(Rv*p)

    @staticmethod
    def RHNumpy(T,qv,P0,PS,hyam,hybm):
        R = np.float32(287.0)
        Rv = np.float32(461.0)
        p = P0 * hyam + PS[:, None] * hybm # Total pressure (Pa)

        T = T.astype(np.float32)
        qv = qv.astype(np.float32)
        p = p.astype(np.float32)

        return Rv*p*qv/(R*ThermLibNumpy.esatNumpy(T))

    @staticmethod
    def qsatNumpy(T,P0,PS,hyam,hybm):
        return ThermLibNumpy.qvNumpy(T,1,P0,PS,hyam,hybm)

    @staticmethod
    def bmse_calc(T,qv,P0,PS,hyam,hybm):
        eps = 0.622 # Ratio of molecular weight(H2O)/molecular weight(dry air)
        R_D = 287 # Specific gas constant of dry air in J/K/kg
        Rv = 461
        # Calculate kappa
        QSAT0 = ThermLibNumpy.qsatNumpy(T,P0,PS,hyam,hybm)
        kappa = 1+(L_V**2)*QSAT0/(Rv*C_P*(T**2))
        # Calculate geopotential
        r = qv/(qv**0-qv)
        Tv = T*(r**0+r/eps)/(r**0+r)
        p = P0 * hyam + PS[:, None] * hybm
        p = p.astype(np.float32)
        RHO = p/(R_D*Tv)
        Z = -cumulative_trapezoid(x=p,y=1/(G*RHO),axis=1)
        Z = np.concatenate((0*Z[:,0:1]**0,Z),axis=1)
        # Assuming near-surface is at 2 meters
        Z = (Z-Z[:,[29]])+2
        # Calculate MSEs of plume and environment
        Tile_dim = [1,30]
        h_plume = np.tile(np.expand_dims(C_P*T[:,-1]+L_V*qv[:,-1],axis=1),Tile_dim)
        h_satenv = C_P*T+L_V*qv+G*Z
        return (G/kappa)*(h_plume-h_satenv)/(C_P*T)


# Physical rescalings: each one undoes the normalization of the raw inputs,
# computes the physical variable, and normalizes it again with its own constants.
# Specific humidity to relative humidity
class QV2RHNumpy:
    def __init__(self, inp_sub, inp_div, inp_subRH, inp_divRH, hyam, hybm):
        self.inp_sub, self.inp_div, self.inp_subRH, self.inp_divRH, self.hyam, self.hybm = \
            np.array(inp_sub), np.array(inp_div), np.array(inp_subRH), np.array(inp_divRH), \
        np.array(hyam), np.array(hybm)
        # Define variable indices here
        self.QBP_idx = slice(0,30)
        self.TBP_idx = slice(30,60)
        self.PS_idx = 60
    def process(self,X):
        Tprior = X[:,self.TBP_idx]*self.inp_div[self.TBP_idx]+self.inp_sub[self.TBP_idx]
        qvprior = X[:,self.QBP_idx]*self.inp_div[self.QBP_idx]+self.inp_sub[self.QBP_idx]
        PSprior = X[:,self.PS_idx]*self.inp_div[self.PS_idx]+self.inp_sub[self.PS_idx]
        RHprior = (ThermLibNumpy.RHNumpy(Tprior,qvprior,P0,PSprior,self.hyam,self.hybm)-\
                    self.inp_subRH[self.QBP_idx])/self.inp_divRH[self.QBP_idx]
        X_result = np.concatenate([RHprior.astype(np.float32),X[:,30:]], axis=1)
        return X_result

# Temperature to moist static energy-conserving plume buoyancy
class T2BMSENumpy:
    def __init__(self, inp_sub, inp_div, inp_subT, inp_divT, hyam, hybm):
        self.inp_sub, self.inp_div, self.inp_subT, self.inp_divT, self.hyam, self.hybm = \
            np.array(inp_sub), np.array(inp_div), np.array(inp_subT), np.array(inp_divT), \
        np.array(hyam), np.array(hybm)
        self.QBP_idx = slice(0,30)
        self.TBP_idx = slice(30,60)
        self.PS_idx = 60
    def process(self,X):
        Tprior = X[:,self.TBP_idx]*self.inp_div[self.TBP_idx]+self.inp_sub[self.TBP_idx]
        qvprior = X[:,self.QBP_idx]*self.inp_div[self.QBP_idx]+self.inp_sub[self.QBP_idx]
        PSprior = X[:,self.PS_idx]*self.inp_div[self.PS_idx]+self.inp_sub[self.PS_idx]
        Bmse = ThermLibNumpy.bmse_calc(Tprior,qvprior,P0,PSprior,self.hyam,self.hybm)
        Bmseprior = (Bmse-self.inp_subT[self.TBP_idx])/self.inp_divT[self.TBP_idx]
        post = np.concatenate([X[:,:30],Bmseprior.astype(np.float32),X[:,60:]], axis=1)
        return post

# Surface latent heat flux to the flux divided by the near-surface moisture disequilibrium
class LHF2LHF_nsDELQNumpy:
    def __init__(self, inp_sub, inp_div, inp_subLHF, inp_divLHF, hyam, hybm, epsilon):
        self.inp_sub, self.inp_div, self.inp_subLHF, self.inp_divLHF, self.hyam, self.hybm, self.epsilon = \
        np.array(inp_sub), np.array(inp_div), \
        np.array(inp_subLHF), np.array(inp_divLHF), \
        np.array(hyam), np.array(hybm),np.array(epsilon)
        self.QBP_idx = slice(0,30)
        self.TBP_idx = slice(30,60)
        self.PS_idx = 60
        self.LHFLX_idx = 63
    def process(self,X):
        qvprior = X[:,self.QBP_idx]*self.inp_div[self.QBP_idx]+self.inp_sub[self.QBP_idx]
        Tprior = X[:,self.TBP_idx]*self.inp_div[self.TBP_idx]+self.inp_sub[self.TBP_idx]
        PSprior = X[:,self.PS_idx]*self.inp_div[self.PS_idx]+self.inp_sub[self.PS_idx]
        LHFprior = X[:,self.LHFLX_idx]*self.inp_div[self.LHFLX_idx]+self.inp_sub[self.LHFLX_idx]

        Qdenprior = (ThermLibNumpy.qsatNumpy(Tprior,P0,PSprior,self.hyam,self.hybm))[:,-1]-qvprior[:,-1]
        Qdenprior = np.maximum(self.epsilon,Qdenprior)

        LHFscaled = (LHFprior/(L_V*Qdenprior)-\
                     self.inp_subLHF[self.LHFLX_idx])/self.inp_divLHF[self.LHFLX_idx]
        LHFtile = np.expand_dims(LHFscaled.astype(np.float32),axis=1)

        post = np.concatenate([X[:,:self.LHFLX_idx],LHFtile,\
                               X[:,(self.LHFLX_idx+1):]],axis=1)
        return post

In [ ]:
# Source code for the custom data generator: run to proceed and read it if you are curious
class InputNormalizer:
    """Subtracts the mean and divides by max(range, std) of each variable."""
    def __init__(self, norm_fn, var_list):
        # Read the normalization constants, then close the file
        with netCDF4.Dataset(norm_fn) as ds:
            names = np.array(ds['var_names'][:])
            names_single = np.array(ds['var_names_single'][:])
            mean, vmin, vmax = ds['mean'][:], ds['min'][:], ds['max'][:]
            std_by_var = ds['std_by_var'][:]
        var_idxs = np.concatenate([np.where(names == v)[0] for v in var_list])
        self.sub = mean[var_idxs]
        std = np.zeros(len(var_idxs))
        for v in var_list:
            std[names[var_idxs] == v] = std_by_var[names_single == v]
        self.div = np.maximum(vmax[var_idxs] - vmin[var_idxs], std)

    def transform(self, x):
        return (x - self.sub) / self.div

class DataGeneratorCI(torch.utils.data.Dataset):
    """
    Each item is one batch (X, Y) of normalized, optionally rescaled, samples.
    Adapted from https://github.com/raspstephan/CBRAIN-CAM/blob/master/cbrain/data_generator.py
    """
    def __init__(self, data_fn, input_vars, output_vars, norm_fn,
                 batch_size=1024, hyam=None, hybm=None,
                 Qscaling=None, Tscaling=None, LHFscaling=None,
                 inp_sub_Qscaling=None, inp_div_Qscaling=None,
                 inp_sub_Tscaling=None, inp_div_Tscaling=None,
                 inp_sub_LHFscaling=None, inp_div_LHFscaling=None,
                 epsQ=1e-3):
        self.batch_size = batch_size
        self.Qscaling, self.Tscaling, self.LHFscaling = Qscaling, Tscaling, LHFscaling

        # Normalization constants of the raw inputs: mean and max(range, std)
        self.input_transform = InputNormalizer(norm_fn, input_vars)
        self.inp_sub, self.inp_div = self.input_transform.sub, self.input_transform.div

        # Open the data file with netCDF4: reading one batch at a time is much faster than via xarray
        self.data_fn = data_fn
        self.data_ds = netCDF4.Dataset(data_fn)
        var_names = self.data_ds['var_names'][:]
        self.input_idxs = np.concatenate([np.where(var_names == v)[0] for v in input_vars])
        self.output_idxs = np.concatenate([np.where(var_names == v)[0] for v in output_vars])
        self.n_samples = self.data_ds['vars'].shape[0]
        self.n_batches = self.n_samples // self.batch_size

        # Physical rescalings
        if Qscaling == 'RH':
            self.QLayer = QV2RHNumpy(self.inp_sub, self.inp_div, inp_sub_Qscaling, inp_div_Qscaling, hyam, hybm)
        if Tscaling == 'BMSE':
            self.TLayer = T2BMSENumpy(self.inp_sub, self.inp_div, inp_sub_Tscaling, inp_div_Tscaling, hyam, hybm)
        if LHFscaling == 'LHF_nsDELQ':
            self.LHFLayer = LHF2LHF_nsDELQNumpy(self.inp_sub, self.inp_div, inp_sub_LHFscaling, inp_div_LHFscaling, hyam, hybm, epsQ)

    def __len__(self):
        return self.n_batches

    def __getitem__(self, index):
        # Grab one batch from the data and split it into inputs and outputs
        start_idx = index * self.batch_size
        batch = self.data_ds['vars'][start_idx:start_idx + self.batch_size]
        X = batch[:, self.input_idxs]
        Y = batch[:, self.output_idxs]

        # Normalize, then rescale; each rescaling starts from the normalized raw inputs
        X_norm = self.input_transform.transform(X)
        X_result = np.copy(X_norm)
        if self.Qscaling:
            X_result = self.QLayer.process(X_result)
        if self.Tscaling:
            X_resultT = self.TLayer.process(X_norm)
            X_result = np.concatenate([X_result[:, :30], X_resultT[:, 30:60], X_result[:, 60:]], axis=1)
        if self.LHFscaling:
            X_resultLHF = self.LHFLayer.process(X_norm)
            X_result = np.concatenate([X_result[:, :60], X_resultLHF[:, 60:]], axis=1)
        return torch.from_numpy(X_result.astype(np.float32)), torch.from_numpy(Y.astype(np.float32))

## Part III: Exposing the Generalization Problem of Standard Neural Networks

The first step, which does not even require anything custom, is to show that neural network subgrid parameterizations trained on cold simulation output struggle to generalize to warm simulation output.

✍ To practice, we will still use the custom data generator class `DataGeneratorCI` and choose the arguments so as not to apply any physical rescaling.

In [ ]:
input_vars = ['QBP','TBP','PS','SOLIN','SHFLX','LHFLX'] # inputs/features
output_vars = ['TPHYSTND'] # outputs/targets

# Training generator loading data from the cold climate
train_gen_cold = DataGeneratorCI(
    data_fn = cold_open, # path to cold climate data
    input_vars = input_vars,
    output_vars = output_vars,
    norm_fn = norm_raw_open # input normalization constants
    )

## **Q3) Create a standard neural network using the hyperparameters of your choice**

We'll refer to this standard (non physically-rescaled) neural network as "brute-force" (BF). The solution gives suggested parameters but we encourage you to be creative here 🎨

Given the environment of this notebook, we highly encourage you to use [`nn.Sequential`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html) to design your neural network. [PyTorch has helpful tutorials and an extensive documentation if you're not yet used to training neural networks.](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html)

In [ ]:
# Be creative here: the network takes 64 inputs and must end with 30 outputs
NN_BF = nn.Sequential(
    nn.Linear(64, ___), ___,    # layer 1 and its activation function
    nn.Linear(___, ___), ___,   # layer 2 and its activation function
    ___,                        # ... as many layers as you like
    nn.Linear(___, 30)          # we need to end up with 30 outputs
)

In [ ]:
# Run this cell to check whether the neural network's architecture
# corresponds to what you had in mind
print(NN_BF)
print(sum(p.numel() for p in NN_BF.parameters()), "trainable parameters")

:::{admonition} **A possible solution for Q3**
:class: note dropdown

The cells below reuse `NN_model` to build the other two networks, so define it in your notebook if you wrote your own architecture.

```python
# 1) Create a generic NN model with 7 layers of 128 neurons and
# Leaky Rectified Linear Unit activation functions
def NN_model(n_inputs=64, n_outputs=30, width=128, n_layers=7):
    layers, n_in = [], n_inputs
    for _ in range(n_layers):
        layers += [nn.Linear(n_in, width), nn.LeakyReLU(0.3)]
        n_in = width
    model = nn.Sequential(*layers, nn.Linear(width, n_outputs))
    # Glorot-uniform weights and zero biases, the Keras defaults this exercise was designed with
    for layer in model:
        if isinstance(layer, nn.Linear):
            nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)
    return model

# 2) Create an instance of this generic NN for the brute-force model
NN_BF = NN_model() # Brute-force model
```
:::

## **Q4) Choose a loss function and an optimizer that are adapted for multivariate regression problems**

:::{admonition} **Hints**
:class: seealso dropdown

1. [Here's a list](https://docs.pytorch.org/docs/stable/optim.html#algorithms) of the optimizers in `torch.optim`. Our go-to was the [Adam optimizer](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html), but the best optimizer will depend on your model's architecture and loss function.
2. [Here's a list](https://docs.pytorch.org/docs/stable/nn.html#loss-functions) of the loss functions in `torch.nn`. Our go-to was the [mean-squared error](https://docs.pytorch.org/docs/stable/generated/torch.nn.MSELoss.html) (a safe bet for multivariate regression problems with well-behaved distributions) but nothing prevents you from being more creative 🎨

PyTorch has no single `.compile()` method like Keras: the loss function and the optimizer are two separate objects, and the optimizer is told which parameters to update.
:::

In [ ]:
mse = nn.___()
optimizer_BF = torch.optim.___(NN_BF.___())

:::{admonition} **A possible solution for Q4**
:class: note dropdown

```python
mse = nn.MSELoss() # Instantiate an unweighted MSE loss
optimizer_BF = torch.optim.Adam(NN_BF.parameters())
```
:::

Now it's time for a little trick 🎪 PyTorch has no built-in `.fit()` method, so below we write the training loop once and reuse it for every network. After every epoch it measures the loss on the warm climate dataset, which tells us whether training our neural network is helping or harming its ability to make predictions in the warm climate. Simply run the cell to create the function `fit`.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on", device)

def make_loader(gen, shuffle):
    # batch_size=None: the generator already returns whole batches
    return torch.utils.data.DataLoader(gen, batch_size=None, shuffle=shuffle)

def fit(model, optimizer, loss_fn, train_gen, warm_gen, epochs):
    """Trains on the cold climate and evaluates on the warm climate after every epoch."""
    model.to(device)
    history = {'loss': [], 'Generalization to warm climate_loss': []}
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for X, Y in make_loader(train_gen, shuffle=True):
            X, Y = X.to(device), Y.to(device)
            loss = loss_fn(model(X), Y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item()
        history['loss'].append(total / len(train_gen))

        model.eval()
        total = 0.0
        with torch.no_grad():
            for X, Y in make_loader(warm_gen, shuffle=False):
                total += loss_fn(model(X.to(device)), Y.to(device)).item()
        history['Generalization to warm climate_loss'].append(total / len(warm_gen))
        print(f"epoch {epoch + 1}/{epochs}: cold loss {history['loss'][-1]:.3e}, "
              f"warm loss {history['Generalization to warm climate_loss'][-1]:.3e}")
    return history

We can now use `fit` to track the loss on the warm simulation output. Each batch is a whole item of the generator, so the `DataLoader` only shuffles the order of the batches, and the warm generator is read in order.

In [ ]:
# Validation generator loading data from the warm climate
valid_gen_warm = DataGeneratorCI(
    data_fn = warm_open, # path to warm climate data
    input_vars = input_vars,
    output_vars = output_vars,
    norm_fn = norm_raw_open # input normalization constants
    )

## **Q5) Train a "brute-force" neural net and track its loss on both the cold and warm data**

⚠ There is a high risk of overfitting the (cold) training set if your neural network has a lot of trainable parameters (weights/biases) and you train it for too many epochs ⚠

Because of time limitations, we are not optimally using the information from the cold climate. Best practices would recommend splitting the cold climate into several training/validation folds that we would use to avoid overfitting and optimize the hyperparameters, including the epoch used to save the best weights and biases for the neural network.

In our [preprint](https://arxiv.org/abs/2112.08440), we show that even if one is careful not to overfit the training set in the cold climate, "brute-force" neural networks will still fail to generalize to much warmer climates (8K here).

In [ ]:
# Complete the code below to train your brute-force neural network
Nep = __ # Choose your number of epochs depending on the time you have (e.g., 10)
history_BF = fit(NN_BF, ___, ___, ___, ___, epochs=Nep)

:::{admonition} **A possible solution for Q5**
:class: note dropdown

```python
Nep = 10 # Number of epochs
history_BF = fit(NN_BF, optimizer_BF, mse, train_gen_cold, valid_gen_warm, epochs=Nep)
```
:::

Training the brute-force network for 10 epochs took about a minute and a half on a laptop CPU, with no GPU; on a GPU it is faster. Each epoch reads the whole 2.4-million-sample cold climate once and the whole warm climate once for the evaluation.

Let's now compare the losses in the cold and the warm climate. They may be so different that we use a log scale on the right-hand side panel.

In [ ]:
fig,ax = plt.subplots(1,2,figsize=(12.5,5))

cold_BFloss = history_BF['loss']
warm_BFloss = history_BF['Generalization to warm climate_loss']
epoch = np.arange(1,len(cold_BFloss)+1)

for iplot in range(2):
  ax[iplot].plot(epoch,cold_BFloss,color='b')
  ax[iplot].scatter(epoch,cold_BFloss,color='b',label='Cold')
  ax[iplot].plot(epoch,warm_BFloss,color='r')
  ax[iplot].scatter(epoch,warm_BFloss,color='r',label='Warm')
  if iplot==1:
    plt.yscale('log');
    ax[iplot].set_ylabel('Log scale')
    ax[iplot].yaxis.set_label_position("right")
    ax[iplot].yaxis.tick_right()
  else: ax[iplot].set_ylabel('Lin scale')
  ax[iplot].set_xlabel('Number of Epochs')

plt.legend()

plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
plt.suptitle('User-chosen loss based on subgrid heating in units [K/s]');

As we can see, our brute-force network struggles to generalize from the cold (blue) to the warm (red) climate, except for very specific choices of architectures. This suggests that the mapping learned by the neural network is **not** climate-invariant, as it cannot transfer from the cold to the warm climate.

Two runs of this exercise with different random seeds gave the same picture. After 10 epochs, the loss on the warm climate was 100 to 3,000 times the loss on the cold climate that the network was trained on, and it jumped by orders of magnitude from one epoch to the next. How badly it fails varies with the initialization, so your curve will differ.

![Brute-force mapping versus climate-invariant mapping](_static/5.3-brute-force-mapping.png)

In the next part and inspired by your earlier result in Q2, we will explore how rescaling specific humidity to relative humidity can help your neural networks generalize across climates.

## Part IV: Physically Rescaling the Inputs to Facilitate Generalization across Climates

![Rescaling the inputs and outputs turns a brute-force mapping into a climate-invariant one](_static/5.3-rescaling-schematic.png)

*The generalization issue we just observed suggests that the "brute-force" mapping from raw inputs to raw outputs is not the same in the cold and in the warm climate, leading to a generalization problem. Can we find a way to rescale our model's inputs so that the mapping learned in the cold climate also applies in the warm climate ("climate invariance")?*

Fig 1 from [our 2021 preprint on climate-invariant machine learning](https://arxiv.org/abs/2112.08440).

We can use our same `DataGeneratorCI` class to create a data generator that automatically rescale inputs before feeding them to the neural network during training. All we have to do is specify a few more arguments, including the names of the rescalings (which are strings):

*   The specific humidity rescaling `Qscaling`
*   The temperature rescaling `Tscaling`
*   The latent heat fluxes rescaling `LHFscaling`

and a much trickier aspect, which is renormalization of the inputs. For that, we change the mean we **sub**tract and the range we **div**ide by through the following arguments:

*   `inp_sub_Qscaling` and `inp_div_Qscaling` for specific humidity
*   `inp_sub_Tscaling` and `inp_div_Tscaling` for temperature
*   `inp_sub_LHFscaling` and `inp_div_LHFscaling` for surface latent heat fluxes

These arguments take in values from separate normalization files, which are still calculated over the cold training set following best practices. We load these normalization files into normalizers below, one per rescaled input.

In [ ]:
# Build the normalizers for our most successful input rescalings:
# Specific humidity to relative humidity
# Temperature to moist static energy-conserving plume buoyancy
# Latent heat fluxes to the version divided by near-surface moisture disequilibrium
normRH = InputNormalizer(RH_open, ['RH', 'TBP', 'PS', 'SOLIN', 'SHFLX', 'LHFLX'])
normBMSE = InputNormalizer(B_open, ['QBP', 'BMSE', 'PS', 'SOLIN', 'SHFLX', 'LHFLX'])
normLHFns = InputNormalizer(LHFnsDELQ_open, ['QBP', 'TBP', 'PS', 'SOLIN', 'SHFLX', 'LHF_nsDELQ'])

Below are the custom data generators loading data from the cold and warm climate and rescaling specific humidity to relative humidity before feeding the inputs to the neural networks 🤖

In [ ]:
# Generator loading data from the cold climate
# and rescaling specific to relative humidity
train_gen_cold_RH = DataGeneratorCI(
    data_fn = cold_open, # path to cold climate data
    input_vars = input_vars,
    output_vars = output_vars,
    norm_fn = norm_raw_open, # input normalization constants
    hyam=hyam, hybm=hybm, # Arrays to define mid-levels of the hybrid vertical coordinate
    Qscaling = 'RH', # Rescale specific humidity to relative humidity,
    inp_sub_Qscaling = normRH.sub, # Constants of the rescaled input
    inp_div_Qscaling = normRH.div
    )

# Generator loading data from the warm climate
# and rescaling specific to relative humidity
valid_gen_warm_RH = DataGeneratorCI(
    data_fn = warm_open, # path to warm climate data
    input_vars = input_vars,
    output_vars = output_vars,
    norm_fn = norm_raw_open, # input normalization constants
    hyam=hyam, hybm=hybm, # Arrays to define mid-levels of hybrid vertical coordinate
    Qscaling = 'RH', # Rescale specific humidity to relative humidity,
    inp_sub_Qscaling = normRH.sub, # Constants of the rescaled input
    inp_div_Qscaling = normRH.div
    )

## **Q6) Repeat the steps of part III to create the climate-invariant counterpart of your brute-force network**

🍎 For comparison purposes, we recommend keeping the exact same architectures and losses as in part III, changing only the data generator. More generally, all hyperparameters (including the number of epochs for training) should be held fixed except for the physical rescaling in the data generator. 🍏

Note: The training should be much slower (by a factor 2-4) now that we have added the physical rescaling in the data generator.  🚆

In [ ]:
# 1) Create a new NN with the exact same hyperparameters
NN_RH = ___

# 2) Create a new optimizer for this neural network (the loss function stays the same)
optimizer_RH = ___

In [ ]:
# 3) Train the (Q to RH)-rescaled NN for the same number of epochs
# but be careful to CHANGE THE DATA GENERATORS TO THE (Q to RH) RESCALING
Nep = 10 # Choose the same number of epochs
history_RH = fit(___, ___, ___, ___, ___, epochs=Nep)

:::{admonition} **A possible solution for Q6 (1/2)**
:class: note dropdown

```python
# 1) Create an instance of our generic NN class for the (Q to RH)-rescaled NN
NN_RH = NN_model()

# 2) Create the optimizer for this neural network
optimizer_RH = torch.optim.Adam(NN_RH.parameters())
```
:::

:::{admonition} **A possible solution for Q6 (2/2)**
:class: note dropdown

```python
# 3) Train the (Q to RH)-rescaled NN for the same number of epochs
Nep = 10 # Choose the same number of epochs
history_RH = fit(NN_RH, optimizer_RH, mse, train_gen_cold_RH, valid_gen_warm_RH, epochs=Nep)
```
:::

Let's repeat the exact same comparison in the cold and warm climates.

In [ ]:
fig,ax = plt.subplots(1,2,figsize=(12.5,5))

cold_RHloss = history_RH['loss']
warm_RHloss = history_RH['Generalization to warm climate_loss']
epoch = np.arange(1,len(cold_RHloss)+1)

for iplot in range(2):
  # Change label depending on iplot
  if iplot==0:
    lin_label_BF = 'Brute Force'; lin_label_CI = 'RH input rescaling';
    scat_label_c = ''; scat_label_w = '';
  else:
    lin_label_BF = ''; lin_label_CI = '';
    scat_label_c = 'Cold'; scat_label_w = 'Warm';
  #
  ax[iplot].plot(epoch,cold_BFloss,color='b', linestyle='--', label=lin_label_BF)
  ax[iplot].scatter(epoch,cold_BFloss,color='b',s=siz/4)
  ax[iplot].plot(epoch,warm_BFloss,color='r',linestyle='--')
  ax[iplot].scatter(epoch,warm_BFloss,color='r',s=siz/4)
  ax[iplot].plot(epoch,cold_RHloss,color='b', label=lin_label_CI)
  ax[iplot].scatter(epoch,cold_RHloss,color='b', label=scat_label_c)
  ax[iplot].plot(epoch,warm_RHloss,color='r')
  ax[iplot].scatter(epoch,warm_RHloss,color='r', label=scat_label_w)
  if iplot==1:
    plt.yscale('log');
    ax[iplot].set_ylabel('Log scale')
    ax[iplot].yaxis.set_label_position("right")
    ax[iplot].yaxis.tick_right()
  else: ax[iplot].set_ylabel('Lin scale')
  ax[iplot].set_xlabel('Number of Epochs')

  ax[iplot].legend()

plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
plt.suptitle('User-chosen loss based on subgrid heating in units [K/s]');

Normally, you should already see a significant improvement: By reducing the amount of extrapolation in the most out-of-distribution input (specific humidity), we can already create a much more generalizable model! 🌈

Making neural networks more generalizable and trustworthy is key to facilitating their deployment in science and operations.

In two runs with different seeds, the warm-climate loss of the network fed with relative humidity ended 5 to 7 times above its cold-climate loss, against 100 to 3,000 times for the brute-force network, and it no longer jumped between epochs. Training took about 2 minutes on a laptop CPU, against 1.5 minutes for the brute-force network.

If you are curious and would like to go further than this, this notebook's last part explores two additional rescalings to make the neural network almost fully "climate-invariant":



1.   Temperature into moist static energy-conserving plume buoyancy,
2.   Latent heat fluxes into latent heat fluxes divided by the near-surface moisture disequilibrium.

The physical assumptions and mathematical details of these rescalings can be found in the appendices of our preprint [at this link](https://arxiv.org/abs/2112.08440).

## Part V (Bonus): Towards "Climate-Invariant" Neural Networks

In [ ]:
# Generator loading data from the cold climate
# and rescaling specific to relative humidity
# temperature into MSE-conserving plume buoyancy
# and latent heat fluxes into LHF divided by near-surface moisture disequilibrium
train_gen_cold_CI = DataGeneratorCI(
    data_fn = cold_open, # path to cold climate data
    input_vars = input_vars,
    output_vars = output_vars,
    norm_fn = norm_raw_open, # input normalization constants
    hyam=hyam, hybm=hybm, # Arrays to define mid-levels of the hybrid vertical coordinate
    Qscaling = 'RH', # Rescale specific humidity to relative humidity,
    Tscaling = 'BMSE', # Rescale temperature to plume buoyancy,
    LHFscaling = 'LHF_nsDELQ', # Rescale latent heat fluxes
    inp_sub_Qscaling = normRH.sub, inp_div_Qscaling = normRH.div, # Constants of the rescaled inputs
    inp_sub_Tscaling = normBMSE.sub, inp_div_Tscaling = normBMSE.div,
    inp_sub_LHFscaling = normLHFns.sub, inp_div_LHFscaling = normLHFns.div
    )

# Generator loading data from the warm climate
# and applying the same three rescalings as above
valid_gen_warm_CI = DataGeneratorCI(
    data_fn = warm_open, # path to warm climate data
    input_vars = input_vars,
    output_vars = output_vars,
    norm_fn = norm_raw_open, # input normalization constants
    hyam=hyam, hybm=hybm, # Arrays to define mid-levels of hybrid vertical coordinate
    Qscaling = 'RH', # Rescale specific humidity to relative humidity,
    Tscaling = 'BMSE', # Rescale temperature to plume buoyancy,
    LHFscaling = 'LHF_nsDELQ', # Rescale latent heat fluxes
    inp_sub_Qscaling = normRH.sub, inp_div_Qscaling = normRH.div, # Constants of the rescaled inputs
    inp_sub_Tscaling = normBMSE.sub, inp_div_Tscaling = normBMSE.div,
    inp_sub_LHFscaling = normLHFns.sub, inp_div_LHFscaling = normLHFns.div
    )

In [ ]:
# 1) Create an instance of our generic NN class for the climate-invariant NN
NN_CI = NN_model()

# 2) Create the optimizer for this neural network
optimizer_CI = torch.optim.Adam(NN_CI.parameters())

# 3) Train the climate-invariant NN for the same number of epochs
Nep = 10 # Choose the same number of epochs
history_CI = fit(NN_CI, optimizer_CI, mse, train_gen_cold_CI, valid_gen_warm_CI, epochs=Nep)

In [ ]:
fig,ax = plt.subplots(1,2,figsize=(12.5,5))

cold_CIloss = history_CI['loss']
warm_CIloss = history_CI['Generalization to warm climate_loss']
epoch = np.arange(1,len(cold_CIloss)+1)

for iplot in range(2):
  # Change label depending on iplot
  if iplot==0:
    lin_label_BF = 'Brute Force'; lin_label_CI = 'Climate Invariant';
    scat_label_c = ''; scat_label_w = '';
  else:
    lin_label_BF = ''; lin_label_CI = '';
    scat_label_c = 'Cold'; scat_label_w = 'Warm';
  #
  ax[iplot].plot(epoch,cold_BFloss,color='b', linestyle='--', label=lin_label_BF)
  ax[iplot].scatter(epoch,cold_BFloss,color='b',s=siz/4)
  ax[iplot].plot(epoch,warm_BFloss,color='r',linestyle='--')
  ax[iplot].scatter(epoch,warm_BFloss,color='r',s=siz/4)
  ax[iplot].plot(epoch,cold_CIloss,color='b', label=lin_label_CI)
  ax[iplot].scatter(epoch,cold_CIloss,color='b', label=scat_label_c)
  ax[iplot].plot(epoch,warm_CIloss,color='r')
  ax[iplot].scatter(epoch,warm_CIloss,color='r', label=scat_label_w)
  if iplot==1:
    plt.yscale('log');
    ax[iplot].set_ylabel('Log scale')
    ax[iplot].yaxis.set_label_position("right")
    ax[iplot].yaxis.tick_right()
  else: ax[iplot].set_ylabel('Lin scale')
  ax[iplot].set_xlabel('Number of Epochs')

  ax[iplot].legend()

plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
plt.suptitle('User-chosen loss based on subgrid heating in units [K/s]');

By combining these three rescalings, you should be able to get very similar performance metrics in the warm and cold climate, despite the fact that the neural network was only trained in a cold climate 🤩

In the same two runs, the climate-invariant network trained on the cold climate reached a loss on the warm climate within 10% of its cold-climate loss after 10 epochs. This network took about 8 minutes per 10 epochs on a laptop CPU, five times longer than the brute-force one, because the data generator now computes the saturation vapor pressure, the plume buoyancy and the humidity deficit for every batch.

🔥 Congratulations: You can now use your physical knowledge to create machine learning algorithms that are robust to changes in their inputs!

And with that, you are officially done with the exercises for ANNs: Well-done! 🎉